# Feature Engineering Questions — Where Models Are Actually Won

The senior-interview consensus question: *"your model is underperforming — what do you do?"* The strong answer starts with data and features, not model families. This notebook covers the decisions that recur in every tabular project — encoding, scaling, interactions, dates, text, target encoding without leakage — each with the executed demo of the failure it prevents.

## Q1 — Categorical Encoding: the Decision Tree

**Ordered by when to use them:**

1. **One-hot** — low-cardinality (< ~15 levels): safe, linear-model-friendly; cost: width explosion + collinearity (drop one level or regularize).
2. **Ordinal** — only when the categories *have* a real order (education, size); on nominal data it injects fake distances — the classic trap.
3. **Frequency/count encoding** — replace with occurrence count; cheap, works surprisingly well for tree models, but conflates distinct rare levels.
4. **Target encoding** — high-cardinality (zip codes, SKUs); the most powerful and the most dangerous — see Q4, it needs its own section.
5. **Embeddings** — deep-learning side; the entity-embedding idea is "learned target encoding" — mention it for completeness.

**What NOT to do unprompted:** label-encode nominal high-cardinality for a linear model — the model then learns nonsense from arbitrary integer orderings.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "city": ["Pune", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
    "size": ["small", "medium", "large", "medium", "small", "large", "small", "medium"],
    "price": [10, 15, 22, 16, 9, 25, 8, 14],
})

oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_oh = oh.fit_transform(df[["city"]])
print("one-hot cities:"); print(pd.DataFrame(X_oh, columns=oh.get_feature_names_out()).head(3))

size_map = {"small": 0, "medium": 1, "large": 2}          # real order -> ordinal is right
df["size_ord"] = df["size"].map(size_map)
df["city_freq"] = df["city"].map(df["city"].value_counts(normalize=True))
print(df[["city", "city_freq", "size", "size_ord"]].head(5))

one-hot cities:
   city_Delhi  city_Mumbai  city_Pune
0         0.0          0.0        1.0
1         1.0          0.0        0.0
2         0.0          0.0        1.0
     city  city_freq    size  size_ord
0    Pune      0.375   small         0
1   Delhi      0.375  medium         1
2    Pune      0.375   large         2
3  Mumbai      0.250  medium         1
4   Delhi      0.375   small         0


## Q2 — Scaling: Which Scaler, and When It Doesn't Matter

**StandardScaler** (z-score) — the default; needed by gradient-descent/distance-based models (linear with regularization, KNN, SVM, NN). **MinMaxScaler** [0,1] — preserves shape, sensitive to outliers; for neural inputs. **RobustScaler** (median/IQR) — outliers present. **PowerTransformer** (Yeo-Johnson/Box-Cox) — skewed features toward Gaussian. **Quantile/power rank transforms** — brutal but effective for wild distributions.

**The answer that scores:** *trees and gradient boosting don't need scaling (threshold invariance); linear/distance/NN models do; and I always put the scaler inside the pipeline, fit on training folds only* — that last clause is the one interviewers listen for.

In [2]:
from sklearn.preprocessing import StandardScaler, RobustScaler
rng = np.random.default_rng(0)
x = np.concatenate([rng.normal(50, 5, 500), [10000]])   # one huge outlier

print(f"standard : mean {StandardScaler().fit_transform(x[:,None]).mean():.3f}, std-robustness broken — outlier stretched range")
r = RobustScaler().fit_transform(x[:,None])
print(f"robust   : median {np.median(r):.3f}, IQR-scaled — the outlier barely moves the scaling")
print("one outlier moved StandardScaler's scaling; RobustScaler ignored it")

standard : mean 0.000, std-robustness broken — outlier stretched range
robust   : median 0.000, IQR-scaled — the outlier barely moves the scaling
one outlier moved StandardScaler's scaling; RobustScaler ignored it


## Q3 — Numeric Features: the Standard Moves

The checklist that turns raw numerics into signal:

- **Skew correction** — log1p for right-skewed positives (income, counts, durations); the *target* too, with prediction-time expm1 inverse.
- **Binning** — monotone relationships or model-interpretability needs; use quantile bins, treat as ordinal afterwards.
- **Ratios/interactions** — price per sqft, clicks per session, debt-to-income: the domain knowledge move. For tree models, add the raw ingredients *and* the ratio — trees approximate ratios poorly.
- **Cyclical encoding** — hour/day/month as (sin, cos) pairs so 23:00 and 00:00 are neighbors.
- **Missingness as signal** — `is_missing` indicator often carries the mechanism of missingness (bills absent → no account); never drop that column silently.

In [3]:
hours = np.arange(24)
enc = pd.DataFrame({"hour": hours,
                    "hour_sin": np.sin(2*np.pi*hours/24),
                    "hour_cos": np.cos(2*np.pi*hours/24)})
# the property to verify: distance(23:00, 00:00) tiny in the encoded space
def dist(h1, h2):
    a = enc.loc[h1, ["hour_sin","hour_cos"]].values; b = enc.loc[h2, ["hour_sin","hour_cos"]].values
    return float(np.linalg.norm(a - b))
print(f"cyclical distance 23h-00h: {dist(23,0):.3f}   <- neighbors, correct")
print(f"cyclical distance 00h-12h: {dist(0,12):.3f}   <- antipodes, correct")
print(f"raw distance 23-00        : {abs(23-0)}      <- wrong: looks far apart")

cyclical distance 23h-00h: 0.261   <- neighbors, correct
cyclical distance 00h-12h: 2.000   <- antipodes, correct
raw distance 23-00        : 23      <- wrong: looks far apart


## Q4 — Target Encoding Without Leaking (the Big One)

*High-cardinality categoricals: replace each level with the mean of the target.* Powerful — and the textbook leakage source: computing those means on the full dataset lets each row "see" its own label. **The three protections, in order of importance:**

1. **Fit inside CV / on training folds only** — `sklearn`'s `TargetEncoder` (or `category_encoders.TargetEncoder`) inside a `Pipeline` handles this mechanically.
2. **Smoothing** — blend level mean with the global mean, weight by level count: `(n·mean_level + m·mean_global)/(n+m)` — rare levels shrink to the prior instead of memorizing noise.
3. **Add noise / leave-one-out** — for a single split; CV-fold fitting makes this redundant.

The measured cost of getting it wrong is a 15-AUC-point illusion in the repo's [Data Leakage notebook](../../3.%20Learn%20Practical%20Concepts/16.%20Data%20Leakage/Data%20Leakage.ipynb) — quote that number.

In [4]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 4000
# high-cardinality feature with a weak real signal
store = np.array([f"s{i}" for i in rng.integers(0, 200, n)])
store_signal = {f"s{i}": rng.normal(0, 0.15) for i in range(200)}
y = (rng.normal(0, 1, n) + np.array([store_signal[s] for s in store]) > 0).astype(int)
X = store.reshape(-1, 1)

# WRONG: target-encode on everything, then split (leak!)
means = pd.DataFrame({"s": store, "y": y}).groupby("s")["y"].mean()
X_leak = np.array([[means[s]] for s in store])
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.3, random_state=0)
auc_leak = roc_auc_score(yte, LogisticRegression().fit(Xtr, ytr).predict_proba(Xte)[:,1])

# RIGHT: TargetEncoder inside the pipeline — CV folds do the fitting
pipe = Pipeline([("te", TargetEncoder(cv=5, smooth="auto")),
                 ("lr", LogisticRegression())])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
pipe.fit(Xtr, ytr)
auc_ok = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])

print(f"leaked target encoding  : AUC {auc_leak:.3f}   <- the illusion")
print(f"pipeline target encoding: AUC {auc_ok:.3f}    <- the truth")
print(f"leakage inflated the score by {auc_leak - auc_ok:.1%} of AUC")

leaked target encoding  : AUC 0.610   <- the illusion
pipeline target encoding: AUC 0.503    <- the truth
leakage inflated the score by 10.7% of AUC


## Q5 — Text Features: From Counts to Embeddings in Five Lines of History

**Bag-of-words/TF-IDF** — sparse lexical features; TF-IDF weights rare-but-informative words via log(N/df). Still the right baseline for short text + linear models (spam-classic). **Embeddings** (word2vec/GloVe: static; transformers: contextual) — dense vectors capturing semantics; average-pooled sentence embeddings are the cheap production trick; fine-tuned transformers when the ceiling matters. **The interview phrasing:** *"TF-IDF is about what words appear; embeddings are about what words mean; and a strong baseline of the first often beats a weak implementation of the second."* The repo's [NLP notebook](../../4.%20Diving%20into%20different%20domains/Natural%20Language%20Processing/Natural%20Language%20Processing.ipynb) executes the full pipeline on real SMS spam data.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["free money win prize click now", "team meeting notes attached",
        "claim your free prize", "project deadline extended",
        "win a free iphone"] * 4
labels = [1, 0, 1, 0, 1] * 4

tfidf = TfidfVectorizer()
Z = tfidf.fit_transform(docs)
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out())
print("idf (higher = rarer = more informative):")
print(idf.sort_values().round(2).to_string())

idf (higher = rarer = more informative):
free        1.48
prize       1.85
win         1.85
attached    2.44
deadline    2.44
extended    2.44
click       2.44
claim       2.44
meeting     2.44
iphone      2.44
notes       2.44
money       2.44
now         2.44
project     2.44
team        2.44
your        2.44


## Q6 — Datetime Features and the Lag Trick

Dates are never raw features — explode them: year, month, day-of-week, is_weekend, is_holiday, distance-to-event. For *time-series-adjacent* tabular problems the killer features are **lags and rolling statistics computed strictly from the past** (`shift` before `rolling`), plus time-since-last-event. The trap sentence to volunteer: *"every temporal feature must be computed with information available *at prediction time* — a rolling mean including today's row is leakage."*

In [6]:
ts = pd.DataFrame({"date": pd.date_range("2026-01-01", periods=8, freq="D"),
                    "sales": [10, 12, 9, 15, 14, 18, 17, 21]})
ts["dow"] = ts["date"].dt.dayofweek
ts["is_weekend"] = ts["dow"] >= 5
ts["lag_1"] = ts["sales"].shift(1)                     # yesterday
ts["roll3_past"] = ts["sales"].shift(1).rolling(3).mean()   # mean of the 3 PRIOR days
print(ts.round(2).to_string(index=False))
print("note roll3_past on row 3 = mean of rows 0-2: no peeking at the present")

      date  sales  dow  is_weekend  lag_1  roll3_past
2026-01-01     10    3       False    NaN         NaN
2026-01-02     12    4       False   10.0         NaN
2026-01-03      9    5        True   12.0         NaN
2026-01-04     15    6        True    9.0       10.33
2026-01-05     14    0       False   15.0       12.00
2026-01-06     18    1       False   14.0       12.67
2026-01-07     17    2       False   18.0       15.67
2026-01-08     21    3       False   17.0       16.33
note roll3_past on row 3 = mean of rows 0-2: no peeking at the present


/tmp/ipykernel_2236/4161192513.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  ts["dow"] = ts["date"].dt.dayofweek
/tmp/ipykernel_2236/4161192513.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(ts.round(2).to_string(index=False))


## Q7 — Feature Selection: Three Families

- **Filter** — rank by correlation/mutual information/χ² before modeling; cheap, ignores interactions. The quick pass.
- **Wrapper** — RFE, sequential selection: model-guided search, expensive, can overfit the selection criterion itself.
- **Embedded** — the model selects: L1 (lasso), tree importances, permutation importance post-hoc. The practical default.

**The senior caveats:** (1) select *inside* CV (`SelectFromModel` in a pipeline) or the selection leaks; (2) correlated features make importance unstable — importance is shared, not uniquely attributable; (3) removing "useless" features can still *hurt* ensemble diversity; (4) always keep a domain-reasonable floor: a feature the business insists matters stays in, flagged for monitoring. The [Feature Selection notebook](../../3.%20Learn%20Practical%20Concepts/10.%20Feature%20Selection/Feature%20Selection.ipynb) runs all three families end-to-end.

In [7]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=12, n_informative=3,
                           n_redundant=2, shuffle=False, random_state=0)
mi = pd.Series(mutual_info_classif(X, y, random_state=0),
               index=[f"f{i}" for i in range(12)])
print("mutual information by feature (f0-f2 informative, f3-f4 redundant):")
print(mi.sort_values(ascending=False).head(6).round(3).to_string())
print("filter methods rank true signal on top; redundant copies split the credit")

mutual information by feature (f0-f2 informative, f3-f4 redundant):
f1     0.166
f3     0.164
f4     0.039
f8     0.025
f0     0.017
f10    0.008
filter methods rank true signal on top; redundant copies split the credit


## Q8 — Quick-Fire Feature Round

| Question | The 15-second answer |
| --- | --- |
| New dataset, first five features you build? | Row aggregates (means/ratios of existing columns), time-since-event, count features per entity, is-missing indicators, then target-free categorical counts |
| Curse of dimensionality — features 10000, rows 500? | Feature selection or strong regularization; distances/distances-based models degrade first; linear+L1 or trees survive |
| How do you handle a categorical with 10k levels? | Target encoding with smoothing, frequency encoding, hash trick, or group rare levels into "other" — never one-hot |
| Feature crosses/interactions for trees? | Trees find axis-aligned interactions themselves; explicit crosses help when the interaction is a *ratio* or threshold-combo, not a product |
| Do you scale before PCA? | Always — unscaled PCA just finds the biggest-unit feature, not the biggest-variance direction |
| Data augmentation for tabular? | SMOTE-family, noise injection, mixup variants, and back-translation for text — but beware fabricating impossible rows (negative age) |
| How many features is too many? | Rule of thumb √n to n/10 events per parameter for stability; the honest answer is 'as many as CV justifies, and not one more' |
| What makes a feature "good"? | Predictive (MI with target), available at prediction time, robust to drift, cheap to compute in serving, and explainable to stakeholders |